# Train TinyGlobalDet (<200k params)

Depthwise-separable detector + global pixel attention. Có early stopping và lưu `best.pt`/`last.pt`.

Lưu ý: notebook hiện train student độc lập. YOLO26m teacher chưa được bật vì task-specific teacher checkpoint đã bị xóa; không dùng COCO YOLO26m trực tiếp vì mapping class không khớp 7 class của bài.

In [ ]:
from pathlib import Path
import random, numpy as np, pandas as pd
from PIL import Image, ImageEnhance
import torch, torch.nn.functional as F
from torchvision.ops import nms as torchvision_nms
from torch.utils.data import Dataset, DataLoader
from tinyglobaldet_model import TinyGlobalDet
ROOT=Path.cwd(); DATA=ROOT/'data'
if not DATA.exists():
    for p in ROOT.parents:
        if (p/'data').exists(): ROOT,DATA=p,p/'data'; break
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); NC=7; SLOTS=2; GW,GH=40,30; BATCH=32; MAX_EPOCHS=100; PATIENCE=12
RUN=ROOT/'runs'/'tinyglobaldet'; RUN.mkdir(parents=True,exist_ok=True)
class DS(Dataset):
    def __init__(self,paths,aug=False): self.paths=list(paths); self.aug=aug
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        p=self.paths[i]; im=Image.open(p).convert('RGB').resize((320,240)); boxes=[]
        for line in (DATA/'train'/'labels'/f'{p.stem}.txt').read_text().splitlines():
            z=line.split();
            if len(z)==5: boxes.append([int(z[0]),*map(float,z[1:])])
        if self.aug and random.random()<.5: im=im.transpose(Image.Transpose.FLIP_LEFT_RIGHT); boxes=[[c,1-x,y,w,h] for c,x,y,w,h in boxes]
        if self.aug and random.random()<.2: im=ImageEnhance.Brightness(im).enhance(random.uniform(.85,1.15))
        x=torch.from_numpy(np.asarray(im,dtype=np.float32).transpose(2,0,1)/255.)
        t=torch.zeros(SLOTS,12,GH,GW); used={}
        for c,x0,y0,w,h in boxes:
            gx=min(GW-1,max(0,int(x0*GW))); gy=min(GH-1,max(0,int(y0*GH))); s=used.get((gy,gx),0)
            if s>=SLOTS: continue
            used[(gy,gx)]=s+1; t[s,0,gy,gx]=1; t[s,1:5,gy,gx]=torch.tensor([x0*GW-gx,y0*GH-gy,w,h]); t[s,5+c,gy,gx]=1
        return x,t
paths=sorted((DATA/'train'/'images').glob('*.png')); random.Random(SEED).shuffle(paths); n=int(.2*len(paths))
tr,va=paths[n:],paths[:n]; train_loader=DataLoader(DS(tr,True),BATCH,True,num_workers=0); val_loader=DataLoader(DS(va),BATCH,False,num_workers=0)
model=TinyGlobalDet(NC,SLOTS).to(DEVICE); nparams=sum(p.numel() for p in model.parameters()); print('device',DEVICE,'params',nparams); assert nparams<200_000

In [ ]:
def loss_fn(p,t):
    pos=t[:,:,0]>.5; lo=F.binary_cross_entropy_with_logits(p[:,:,0],t[:,:,0],pos_weight=torch.tensor(3.,device=p.device))
    pbox=p[:,:,1:5].permute(0,1,3,4,2); tbox=t[:,:,1:5].permute(0,1,3,4,2); pcls=p[:,:,5:].permute(0,1,3,4,2); tcls=t[:,:,5:].permute(0,1,3,4,2)
    if pos.any(): lb=F.smooth_l1_loss(torch.sigmoid(pbox[pos]),tbox[pos]); lc=F.binary_cross_entropy_with_logits(pcls[pos],tcls[pos])
    else: lb=p.sum()*0; lc=p.sum()*0
    return lo+5*lb+lc
def box_iou(a,b):
    if len(a)==0 or len(b)==0: return torch.zeros((len(a),len(b)))
    lt=torch.maximum(a[:,None,:2],b[None,:,:2]); rb=torch.minimum(a[:,None,2:],b[None,:,2:]); inter=(rb-lt).clamp(min=0).prod(2); area_a=(a[:,2]-a[:,0])*(a[:,3]-a[:,1]); area_b=(b[:,2]-b[:,0])*(b[:,3]-b[:,1]); return inter/(area_a[:,None]+area_b[None,:]-inter+1e-7)
def decode_one(raw, conf=0.001):
    q=raw.sigmoid().cpu(); boxes=[]; scores=[]; classes=[]
    for s in range(SLOTS):
        for gy in range(GH):
            for gx in range(GW):
                v=q[s,:,gy,gx]; cs,cl=v[5:].max(0); score=float(v[0]*cs)
                if score<conf: continue
                cx=(float(v[1])+gx)/GW; cy=(float(v[2])+gy)/GH; w=float(v[3]); h=float(v[4]); boxes.append([cx-w/2,cy-h/2,cx+w/2,cy+h/2]); scores.append(score); classes.append(int(cl))
    keep=[]; bt=torch.tensor(boxes); st=torch.tensor(scores); ct=torch.tensor(classes)
    for c in set(classes):
        ids=(ct==c).nonzero(as_tuple=False).flatten(); top=ids[st[ids].topk(min(100,len(ids))).indices]; keep.extend(top[torchvision_nms(bt[top],st[top],.5)].tolist())
    return [boxes[i] for i in keep],[scores[i] for i in keep],[classes[i] for i in keep]
def target_one(t):
    boxes=[]; classes=[]
    for s in range(SLOTS):
        for gy in range(GH):
            for gx in range(GW):
                if t[s,0,gy,gx]<.5: continue
                x,y,w,h=t[s,1:5,gy,gx].tolist(); cx=(x+gx)/GW; cy=(y+gy)/GH; boxes.append([cx-w/2,cy-h/2,cx+w/2,cy+h/2]); classes.append(int(t[s,5:,gy,gx].argmax()))
    return boxes,classes
def map50(loader):
    preds={c:[] for c in range(NC)}; gts={c:{} for c in range(NC)}; image_id=0
    with torch.no_grad():
        for x,t in loader:
            raw=model(x.to(DEVICE)).cpu()
            for b in range(len(x)):
                pb,ps,pc=decode_one(raw[b]); gb,gc=target_one(t[b])
                for c in range(NC): gts[c][image_id]=[torch.tensor(g) for g,k in zip(gb,gc) if k==c]
                for box,score,c in zip(pb,ps,pc): preds[c].append((score,image_id,torch.tensor(box)))
                image_id+=1
    aps=[]
    for c in range(NC):
        total=sum(len(v) for v in gts[c].values()); ranked=sorted(preds[c],reverse=True,key=lambda z:z[0]); tp=torch.zeros(len(ranked)); fp=torch.zeros(len(ranked)); used={i:set() for i in gts[c]}
        for j,(_,im,box) in enumerate(ranked):
            gt=gts[c].get(im,[]); best_iou=0.; best_i=-1
            if gt:
                ious=box_iou(box[None],torch.stack(gt))[0]
                for k,v in enumerate(ious):
                    if k not in used[im] and float(v)>best_iou: best_iou=float(v); best_i=k
            if best_iou>=.5: tp[j]=1; used[im].add(best_i)
            else: fp[j]=1
        if total==0: continue
        rec=torch.cumsum(tp,0)/total; prec=torch.cumsum(tp,0)/(torch.cumsum(tp+fp,0)+1e-7); mrec=torch.cat([torch.tensor([0.]),rec,torch.tensor([1.])]); mpre=torch.cat([torch.tensor([1.]),prec,torch.tensor([0.])])
        for j in range(len(mpre)-2,-1,-1): mpre[j]=max(mpre[j],mpre[j+1])
        aps.append(float(((mrec[1:]-mrec[:-1])*mpre[1:]).sum()))
    return float(np.mean(aps)) if aps else 0.
opt=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4); sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,MAX_EPOCHS)
best_map=-1.; bad=0; hist=[]
for ep in range(1,MAX_EPOCHS+1):
    model.train(); tl=0.
    for x,t in train_loader:
        x,t=x.to(DEVICE),t.to(DEVICE); opt.zero_grad(set_to_none=True); l=loss_fn(model(x),t); l.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5); opt.step(); tl+=l.item()*len(x)
    model.eval(); vl=0.
    with torch.no_grad():
        for x,t in val_loader: vl+=loss_fn(model(x.to(DEVICE)),t.to(DEVICE)).item()*len(x)
    tl/=len(train_loader.dataset); vl/=len(val_loader.dataset); val_map50=map50(val_loader); sch.step(); state={'epoch':ep,'model':model.state_dict(),'optimizer':opt.state_dict(),'params':nparams,'val_map50':val_map50}
    torch.save(state,RUN/'last.pt'); hist.append({'epoch':ep,'train_loss':tl,'val_loss':vl,'val_map50':val_map50}); print(f'{ep:03d}: train={tl:.5f} val_loss={vl:.5f} mAP50={val_map50:.5f}')
    if val_map50>best_map+1e-5: best_map,bad=val_map50,0; torch.save(state,RUN/'best.pt')
    else: bad+=1
    if bad>=PATIENCE: print('early stopping by mAP50'); break
pd.DataFrame(hist).to_csv(RUN/'history.csv',index=False); print('best mAP50:',best_map); print('best:',RUN/'best.pt'); print('last:',RUN/'last.pt')